In [2]:
import numpy as np
from surmof_cavity import ag_surmof_cavity_trref as transmission
import matplotlib.pyplot as plt
import matplotlib as mpl
from functools import partial
import contourint
import functools

In [3]:
material_poles = np.array([448.79110491874115, 438.2930673770547, 412.93727009075883])/300*2*np.pi

damping = np.array([6.2, 6.0, 5.3])
gamma_half = damping*np.pi/300

material_poles = np.sqrt(material_poles**2-gamma_half**2)-1j*gamma_half

In [4]:
pad_r = 0.02
pad_i = 0.002j
radius_r = 10
radius_i = 0.8j
punchout = 0.3
overlap = 0.4
num_recursions = 18
pad = 0 #0.002

integration_centers = list(material_poles[-1:]-pad)

centers = integration_centers*num_recursions
# centers = [
#     pole - pad_r - radius_r for pole in material_poles
# ]
# centers += [
#     pole - pad_i - radius_i for pole in material_poles
# ]
# centers += [
#     pole + pad_i + radius_i for pole in material_poles
# ]

num_points = [512]*len(integration_centers)*num_recursions
radii = []
for i in range(num_recursions):
    radii += [(radius_r + radius_i)*((punchout+overlap)**i)]*len(integration_centers)

# contours = [
#     contourint.Ellipse(center=center, radius=radius, num_points=nump) for 
#     center, radius, nump in zip(centers, radii, num_points)
# ]

#contour = contourint.Ellipse(center = 15-1j, radius=14+2j, num_points = 2048)

In [5]:
radii

[(10+0.8j),
 (7+0.5599999999999999j),
 (4.8999999999999995+0.39199999999999996j),
 (3.4299999999999993+0.2743999999999999j),
 (2.4009999999999994+0.19207999999999997j),
 (1.6806999999999994+0.13445599999999996j),
 (1.1764899999999996+0.09411919999999997j),
 (0.8235429999999997+0.06588343999999997j),
 (0.5764800999999997+0.04611840799999998j),
 (0.4035360699999998+0.032282885599999984j),
 (0.2824752489999998+0.02259801991999999j),
 (0.19773267429999988+0.01581861394399999j),
 (0.1384128720099999+0.011073029760799992j),
 (0.09688901040699992+0.007751120832559993j),
 (0.06782230728489994+0.0054257845827919956j),
 (0.047475615099429956+0.0037980492079543967j),
 (0.033232930569600964+0.0026586344455680772j),
 (0.023263051398720674+0.001861044111897654j)]

In [6]:
contours= [
    contourint.Stack(contours=[
        contourint.Ellipse(center=center, radius=np.conj(radius)*punchout, num_points=nump),
        contourint.Ellipse(center=center, radius=radius, num_points=nump)
    ]) for 
    center, radius, nump in zip(centers, radii, num_points)
]

In [7]:
ts = 0.025*(np.arange(50)+2)
all_poles = []
all_residues = []
npoles = 1#3
max_num_poles = 8

for thickness in ts:
    t_poles = []
    t_residues = []
    for contour in contours:
        #functools.lru_cache(maxsize=None)()
        f = partial(transmission, thickness=thickness, npoles=npoles)
        try:
            poles, residues = contourint.locate_poles(contour, f, M=max_num_poles, cutoff=1e-12)
        except ValueError:
            poles = residues = np.array([])
        num_found = len(poles)
        missing = (max_num_poles-num_found)

        poles = np.concatenate([poles, [np.nan]*missing])
        residues = np.concatenate([residues, [np.nan]*missing])
        
        t_poles.append(poles)
        t_residues.append(residues)
    all_poles.append(t_poles)
    all_residues.append(t_residues)

In [ ]:
wfreqr1 = material_poles[-1].real - np.logspace(-6, -0.8, 800) 

wfreqi = np.linspace(-0.05, -0.06, 200)

wfreqr = np.concatenate([wfreqr1 ,np.linspace(-1, 16, 200)])
wfreqr = np.sort(wfreqr)

wfreqi = np.concatenate([wfreqi, np.linspace(-1, 0.2, 200)])
wfreqi = np.sort(wfreqi)

Wfreqr, Wfreqi = np.meshgrid(wfreqr, wfreqi, indexing='ij')
Wfreq = Wfreqr + 1j*Wfreqi

In [ ]:
F = f(Wfreq)

/users/tfp/jdf/code/agsurmof-cavity-rse/notebooks/surmof_cavity.py:39: RuntimeWarning: overflow encountered in exp
  den = 1 - m1ref_back * m2ref_fwd * np.exp(1j*k3*2*d3)
/users/tfp/jdf/code/agsurmof-cavity-rse/notebooks/surmof_cavity.py:39: RuntimeWarning: overflow encountered in multiply
  den = 1 - m1ref_back * m2ref_fwd * np.exp(1j*k3*2*d3)
/users/tfp/jdf/code/agsurmof-cavity-rse/notebooks/surmof_cavity.py:39: RuntimeWarning: invalid value encountered in multiply
  den = 1 - m1ref_back * m2ref_fwd * np.exp(1j*k3*2*d3)
/users/tfp/jdf/code/agsurmof-cavity-rse/notebooks/surmof_cavity.py:40: RuntimeWarning: overflow encountered in exp
  tr = m1tr_fwd * m2tr_fwd * np.exp(1j*k3*d3) / den
/users/tfp/jdf/code/agsurmof-cavity-rse/notebooks/surmof_cavity.py:40: RuntimeWarning: invalid value encountered in multiply
  tr = m1tr_fwd * m2tr_fwd * np.exp(1j*k3*d3) / den
/users/tfp/jdf/code/agsurmof-cavity-rse/notebooks/surmof_cavity.py:40: RuntimeWarning: overflow encountered in divide
  tr = m1t

In [ ]:
tolerances = np.sqrt(np.abs(np.array(radii))) * 1e-3
num_shells = len(radii)
search_shells = 3
min_hits = 2
filtered_all_poles = []
for t_poles in all_poles: # poles of a given thickness
    filtered_t_poles = []
    for (i, outer), tol in zip(enumerate(t_poles), tolerances):
        for reference_pole in outer: # one specific pole
            hits = 0
            for idx_inner_shell in range(min(num_shells-i, search_shells)):
                dist = np.abs(reference_pole - t_poles[i + idx_inner_shell])
                hits += np.any(dist<tol)
            if hits>min_hits:
                filtered_t_poles.append(reference_pole)
    filtered_all_poles.append(filtered_t_poles)

/tmp/ipykernel_1035994/2044969871.py:12: RuntimeWarning: invalid value encountered in subtract
  dist = np.abs(reference_pole - t_poles[i + idx_inner_shell])


In [ ]:
cmap = mpl.cm.viridis
norm = mpl.colors.Normalize(vmin=min(ts), vmax=max(ts))
colors = cmap(norm(ts))

plt.pcolormesh(Wfreqr, Wfreqi, np.abs(F).astype(float), alpha=1, rasterized=True, vmax=1e1)
xlim, ylim = plt.xlim(), plt.ylim()
for contour in contours:
    plt.scatter(contour.points.real, contour.points.imag, s=0.1, alpha = 0.1)

first = True
for color, t_poles in list(zip(colors, filtered_all_poles))[::-1]:
    poles = np.array(t_poles).flatten()
    s = 1.5
    if first:
        first=False
        s=3
        color="k"
    plt.scatter(poles.real, poles.imag, color=color, s=s, alpha = 0.9)

plt.colorbar(mpl.cm.ScalarMappable(cmap=cmap, norm=norm), ax=plt.gca(), label="cavity thickness [$\mu m$]")

plt.xlabel("$\Re\{\omega\}$")
plt.ylabel("$\Im\{\omega\}$")

plt.xlim(xlim)
plt.ylim(ylim)

plt.savefig("out/poles_overview.pdf", dpi=300)


plt.xlim(8, 10)
plt.ylim(-0.12, -0.05)
plt.xlim(8.5, 8.7)
plt.ylim(-0.07, -0.05)
plt.savefig("out/poles_zoom.pdf", dpi=300)

NameError: name 'mpl' is not defined